# MAIN TASK – EVA02 LARGE PATCH14 448 FEATURE CLASSIFICATION

In [57]:
# import packages
import os, gc, time, json
import numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb
import optuna
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [ ]:

# ----------------------------------------------------------
# 0. Setup
# ----------------------------------------------------------
BASE_DIR = os.path.dirname(os.getcwd())  # Use current working directory
DATA_DIR = os.path.join(BASE_DIR, "data", "features")  # Point to local features directory
OUTPUT_DIR = os.path.join(BASE_DIR, "results")  # Use existing results directory
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "main_xgb_checkpoint.json")
META_PATH = os.path.join(OUTPUT_DIR, "main_xgb_meta.json")

# train, val and test file used for all training. In this case test is v2.
TRAIN_FEATURES_FILE = "train_eva02_large_patch14_448.mim_m38m_ft_in22k_in1k.csv"
VAL_FEATURES_FILE = "val_eva02_large_patch14_448.mim_m38m_ft_in22k_in1k.csv"
TEST_FEATURES_FILE = "v2_eva02_large_patch14_448.mim_m38m_ft_in22k_in1k.csv"

TRAIN_CONFIG = {
    'max_rounds': 300,
    'early_stop_rounds': 20,
    'checkpoint_interval': 10
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Check Data Availability
# ----------------------------------------------------------
print("📦 Checking dataset availability...")

required_files = [TRAIN_FEATURES_FILE, VAL_FEATURES_FILE, TEST_FEATURES_FILE]

missing_files = []
for file in required_files:
    file_path = os.path.join(DATA_DIR, file)
    if not os.path.exists(file_path):
        missing_files.append(file)
    else:
        print(f"✅ Found {file}")

if missing_files:
    print(f"❌ Missing files: {missing_files}")
    print("Please ensure the feature CSV files are in the data/features/ directory")
else:
    print("✅ All required feature files found")

In [ ]:
# Device detection function
def detect_device():
    """Detect the best available device for XGBoost training."""
    try:
        # Check for CUDA
        import torch
        if torch.cuda.is_available():
            return 'cuda', 'CUDA GPU'
    except ImportError:
        pass
    
    # Note: XGBoost doesn't support MPS directly, so we use CPU for Apple Silicon
    # but we can still detect MPS availability for information purposes
    mps_available = False
    try:
        import torch
        if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            mps_available = True
    except ImportError:
        pass
    
    # XGBoost only supports: cpu, cuda, gpu
    # For Apple Silicon, we use CPU (XGBoost doesn't have native MPS support)
    if mps_available:
        return 'cpu', f'CPU (Apple Silicon detected, but XGBoost uses CPU)'
    else:
        return 'cpu', 'CPU'

# Detect and set device
DEVICE, DEVICE_NAME = detect_device()
print(f"🚀 Detected device: {DEVICE_NAME}")
print(f"   XGBoost will use: {DEVICE}")

## Device Support

This notebook automatically detects and uses the best available device for XGBoost training:

- **CUDA GPU**: For NVIDIA GPUs (requires CUDA-enabled XGBoost)
- **CPU**: For all systems (including Apple Silicon)

### Installing CUDA-enabled XGBoost

To use CUDA acceleration, install XGBoost with GPU support:

```bash
# Option 1: Using pip
pip install xgboost[gpu]

# Option 2: Using conda
conda install -c conda-forge xgboost-gpu

**Note**: Apple Silicon users will use CPU acceleration as XGBoost doesn't have native MPS support yet.


In [ ]:
# ----------------------------------------------------------
# REUSABLE TRAINING FUNCTION WITH CHECKPOINTING
# ----------------------------------------------------------

def train_xgboost_with_checkpointing(
    X_train, y_train, X_val, y_val,
    params,
    checkpoint_path, meta_path,
    sample_weights=None,
    max_rounds=300,
    checkpoint_interval=10,
    early_stop_rounds=20,
    model_name="XGBoost Model"
):
    """
    Train XGBoost model with DMatrix and full checkpointing support.
    
    Args:
        X_train, y_train: Training data
        X_val, y_val: Validation data
        params: XGBoost parameters dictionary
        checkpoint_path: Path to save model checkpoint
        meta_path: Path to save training metadata
        sample_weights: Optional sample weights for training
        max_rounds: Maximum training rounds
        checkpoint_interval: Save checkpoint every N rounds
        early_stop_rounds: Early stopping patience
        model_name: Name for logging purposes
    
    Returns:
        trained_booster: Trained XGBoost booster
        clf: XGBClassifier for easy prediction
        training_meta: Training metadata
    """
    
    print(f"\n🚀 Starting {model_name} training with DMatrix...")
    print(f"Device: {DEVICE_NAME} ({DEVICE})")
    print(f"Checkpoint: Every {checkpoint_interval} rounds")
    
    # --- Create DMatrix ---
    if sample_weights is not None:
        dtrain = xgb.DMatrix(X_train, label=y_train, weight=sample_weights)
        print(f"✅ Created DMatrix with sample weights (shape: {sample_weights.shape})")
    else:
        dtrain = xgb.DMatrix(X_train, label=y_train)
        print("✅ Created DMatrix without sample weights")
    
    deval = xgb.DMatrix(X_val, label=y_val)
    watchlist = [(dtrain, 'train'), (deval, 'eval')]
    
    # --- Check for existing checkpoint ---
    start_iteration = 0
    booster = None
    cumulative_time = 0
    best_score = float('inf')
    
    if os.path.exists(checkpoint_path) and os.path.exists(meta_path):
        print(f"\n📂 Found existing {model_name} checkpoint, resuming training...")
        try:
            booster = xgb.Booster()
            booster.load_model(checkpoint_path)
            
            with open(meta_path, 'r') as f:
                meta = json.load(f)
                start_iteration = meta.get('iteration', 0)
                cumulative_time = meta.get('cumulative_time', 0)
                best_score = meta.get('best_score', float('inf'))
            
            print(f"✅ Resumed from iteration {start_iteration}")
            print(f"   Previous training time: {cumulative_time/60:.1f} min")
        except Exception as e:
            print(f"⚠️ Failed to load checkpoint: {e}")
            print("   Starting from scratch...")
            booster = None
            start_iteration = 0
    else:
        print("   No checkpoint found, starting fresh training...")
    
    # --- Training loop with checkpointing ---
    start_time = time.perf_counter()
    remaining_rounds = max_rounds - start_iteration
    
    if remaining_rounds > 0:
        class CheckpointCallback(xgb.callback.TrainingCallback):
            def __init__(self, checkpoint_path, meta_path, interval, start_iter, cum_time, model_name):
                self.checkpoint_path = checkpoint_path
                self.meta_path = meta_path
                self.interval = interval
                self.start_iter = start_iter
                self.cum_time = cum_time
                self.model_name = model_name
                self.best_score = float('inf')
                self.best_iteration = 0

            def after_iteration(self, model, epoch, evals_log):
                current_iter = self.start_iter + epoch + 1

                if (epoch + 1) % self.interval == 0:
                    elapsed = time.perf_counter() - start_time
                    total_time = self.cum_time + elapsed

                    if evals_log and 'eval' in evals_log and 'mlogloss' in evals_log['eval']:
                        current_score = evals_log['eval']['mlogloss'][-1]
                        if current_score < self.best_score:
                            self.best_score = current_score
                            self.best_iteration = current_iter

                    model.save_model(self.checkpoint_path)

                    meta = {
                        'iteration': current_iter,
                        'cumulative_time': total_time,
                        'best_score': self.best_score,
                        'best_iteration': self.best_iteration,
                        'model_name': self.model_name,
                        'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
                    }
                    with open(self.meta_path, 'w') as f:
                        json.dump(meta, f, indent=2)

                    print(f"[{time.strftime('%H:%M:%S')}] 💾 {self.model_name} checkpoint saved at iteration {current_iter} ({total_time/60:.1f} min total)")

                return False

        checkpoint_callback = CheckpointCallback(
            checkpoint_path, meta_path, checkpoint_interval, start_iteration, cumulative_time, model_name
        )

        booster = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=remaining_rounds,
            evals=watchlist,
            early_stopping_rounds=TRAIN_CONFIG["early_stop_rounds"],
            verbose_eval=50,
            xgb_model=booster,
            callbacks=[checkpoint_callback]
        )

        duration = time.perf_counter() - start_time
        total_duration = cumulative_time + duration

        print(f"\n✅ {model_name} training completed!")
        print(f"   This session: {duration/60:.1f} min")
        print(f"   Total time: {total_duration/60:.1f} min")
        print(f"   Best iteration: {booster.best_iteration}")

        # Save final model
        booster.save_model(checkpoint_path)
        final_meta = {
            'iteration': start_iteration + remaining_rounds,
            'cumulative_time': total_duration,
            'best_iteration': booster.best_iteration,
            'training_complete': True,
            'model_name': model_name,
            'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
        }
        with open(meta_path, 'w') as f:
            json.dump(final_meta, f, indent=2)

        print(f"💾 Final {model_name} saved to {checkpoint_path}")
    else:
        print(f"✅ {model_name} training already complete!")
        with open(meta_path, 'r') as f:
            meta = json.load(f)
            total_duration = meta.get('cumulative_time', 0)
    
    # --- Create XGBClassifier for easy prediction ---
    clf = XGBClassifier()
    clf.load_model(checkpoint_path)
    
    # --- Load training metadata ---
    with open(meta_path, 'r') as f:
        training_meta = json.load(f)
    
    return booster, clf, training_meta

print("✅ Reusable training function created!")


In [ ]:
# Quick inspect header
import pandas as pd
path = os.path.join(DATA_DIR, TRAIN_FEATURES_FILE)
sample = pd.read_csv(path, nrows=5)
print("Raw columns:", sample.columns.tolist())
sample_clean = sample.loc[:, ~sample.columns.str.contains('^Unnamed')]
print("Clean columns:", sample_clean.columns.tolist())
print("Total clean cols:", sample_clean.shape[1])

In [ ]:
# ----------------------------------------------------------
# 3. Load Datasets with Memory Management
# ----------------------------------------------------------
def load_features_chunked(path, feature_cols, label_col='label', chunk_size=100000, dtype='float32'):
    print(f"📥 Loading {os.path.basename(path)} in chunks...")

    # Expected
    expected_features = len(feature_cols)

    # Select only needed cols (ignore path, unnamed)
    usecols = feature_cols + [label_col]
    dtype_dict = {col: dtype for col in feature_cols}
    dtype_dict[label_col] = 'int32'

    X_chunks = []
    y_chunks = []

    total_rows = 0
    chunk_iter = pd.read_csv(path, usecols=usecols, chunksize=chunk_size, dtype=dtype_dict, on_bad_lines='skip')

    for chunk in chunk_iter:
        # Ensure order: features then label
        if list(chunk.columns[:-1]) != feature_cols or chunk.columns[-1] != label_col:
            chunk = chunk.reindex(columns=feature_cols + [label_col])

        X = chunk[feature_cols].values.astype(dtype)
        y = chunk[label_col].values.astype('int32')

        if X.shape[1] != expected_features:
            raise ValueError(f"Expected {expected_features} features, got {X.shape[1]} in chunk")

        X_chunks.append(X)
        y_chunks.append(y)
        total_rows += len(chunk)
        if total_rows % 500000 == 0:
            print(f"   Loaded {total_rows} rows so far...")

    X = np.concatenate(X_chunks, axis=0)
    y = np.concatenate(y_chunks, axis=0)
    del X_chunks, y_chunks, chunk_iter
    gc.collect()

    mem_usage = X.nbytes / (1024**3)
    print(f"✅ Loaded {total_rows} rows, {expected_features} features + label")
    print(f"   Memory usage (X only): {mem_usage:.2f} GB")
    return X, y

# Define feature columns based on your CSV (0 to 1023 as strings)
FEATURE_COLS = [str(i) for i in range(1024)]

# --- Load Train ---
print("\n" + "="*60)
print("LOADING TRAINING DATA")
print("="*60)
X_train_full, y_train_full = load_features_chunked(
    os.path.join(DATA_DIR, TRAIN_FEATURES_FILE),
    feature_cols=FEATURE_COLS
)
print(f"   X_train_full shape: {X_train_full.shape}, y_train_full shape: {y_train_full.shape}")

print("\n🔀 Creating custom validation split (10% or at least 1000 samples)...")
X_train, X_custom_val, y_train, y_custom_val = train_test_split(
    X_train_full, y_train_full, test_size=max(0.1, 1000 / len(X_train_full)), random_state=42, stratify=y_train_full
)
del X_train_full, y_train_full
gc.collect()
print(f"   Train: {X_train.shape}, Custom Val: {X_custom_val.shape}")

# --- Load Val ---
print("\n" + "="*60)
print("LOADING VALIDATION DATA")
print("="*60)
X_val, y_val = load_features_chunked(
    os.path.join(DATA_DIR, VAL_FEATURES_FILE),
    feature_cols=FEATURE_COLS
)
print(f"   X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")

# --- Load Test ---
print("\n" + "="*60)
print("LOADING TEST DATA")
print("="*60)
X_test, y_test = load_features_chunked(
    os.path.join(DATA_DIR, TEST_FEATURES_FILE),
    feature_cols=FEATURE_COLS
)
print(f"   X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

print("\n✅ All data loaded successfully!")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
# ----------------------------------------------------------
# 4. Hyperparameter Tuning with Optuna
# ----------------------------------------------------------

TUNE_HYPERPARAMS = False  # Set to True if you want tuning, FALSE to use previous best params

if TUNE_HYPERPARAMS:
    print("\n🎯 Running hyperparameter tuning with Optuna...")

    # Sample data
    sample_size = min(20000, len(X_train))
    sample_idx = np.random.choice(len(X_train), sample_size, replace=False)
    X_sample = X_train[sample_idx].astype('float16')
    y_sample = y_train[sample_idx]

    dsample = xgb.DMatrix(X_sample, label=y_sample)

    def objective(trial):
        params = {
            'objective': 'multi:softprob',
            'num_class': 1000,
            'tree_method': 'hist',  # Updated
            'device': DEVICE,        # Use detected device (CUDA/MPS/CPU)
            'eval_metric': 'mlogloss',
            'max_depth': trial.suggest_int('max_depth', 4, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.7, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 5, log=True),
            'seed': 42
        }

        cv_results = xgb.cv(
            params,
            dsample,
            num_boost_round=200,
            nfold=3,
            stratified=True,
            metrics='mlogloss',
            early_stopping_rounds=20,
            seed=42,
            verbose_eval=False
        )

        return cv_results['test-mlogloss-mean'].min()

    try:
        study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
        study.optimize(objective, n_trials=8, timeout=600)

        best_params = study.best_params
        print(f"✅ Best params: {best_params}")
        print(f"   Best mlogloss: {study.best_value:.4f}")

        del X_sample, y_sample, dsample
        gc.collect()
    except Exception as e:
        print(f"⚠️ Tuning failed: {e}")
        best_params = {
            'learning_rate': 0.1,
            'max_depth': 6,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_lambda': 1
        }
else:
    best_params = {
        'learning_rate': 0.12482239745005622,
        'max_depth': 5,
        'subsample': 0.7816177731671292,
        'colsample_bytree': 0.7107888259286985,
        'reg_lambda': 0.6279040428885944
    }
    print(f"Using optimized params from previous run: {best_params}")

In [ ]:
# Check system resources (CPU version)
import psutil
print(f"CPU cores: {psutil.cpu_count()}")
print(f"Available memory: {psutil.virtual_memory().available / (1024**3):.1f} GB")

In [ ]:
# ----------------------------------------------------------
# 5. Main Training with Checkpointing & Auto-Resume
# ----------------------------------------------------------

# --- Memory Reduction: Cast to float32 ---
# Uses single precision for features (XGBoost GPU upcasts internally; safe for embeddings)
print("Casting features to float32...")
X_train = X_train.astype('float32') # Cast original X_train
X_custom_val = X_custom_val.astype('float32')
gc.collect()

# --- Updated Params for Low VRAM ---
# Add memory-efficient settings: fewer bins, shallower/limited trees, aggressive sampling
params = {
    'objective': 'multi:softprob',
    'num_class': 1000,
    'tree_method': 'hist',
    'device': DEVICE,  # Use detected device (CUDA/MPS/CPU)
    'eval_metric': 'mlogloss',
    'eta': best_params.get('learning_rate', 0.1),
    'max_depth': min(best_params.get('max_depth', 6), 6),  # Allow deeper trees
    'subsample': 0.8,  # Restore higher sampling for better performance
    'colsample_bytree': 0.8,  # Restore higher feature sampling
    'reg_lambda': best_params.get('reg_lambda', 1),
    'random_state': 42,
    'seed': 42,
    'max_bin': 256,  # Use default bin size
    'grow_policy': 'lossguide',  # Efficient tree growth
    'max_leaves': 64  # Allow more leaves
}

print("\n🚀 Training XGBoost model with reusable function...")
print(f"Device: {DEVICE_NAME} ({DEVICE})")
print(f"Parameters: {params}")
print(f"Checkpoint: Every {TRAIN_CONFIG["checkpoint_interval"]} rounds")

# --- Use Reusable Training Function ---
booster, clf, training_meta = train_xgboost_with_checkpointing(
    X_train=X_train, y_train=y_train,
    X_val=X_custom_val, y_val=y_custom_val,
    params=params,
    checkpoint_path=CHECKPOINT_PATH,
    meta_path=META_PATH,
    sample_weights=None,  # No sample weights for main training
    max_rounds=TRAIN_CONFIG["max_rounds"],
    checkpoint_interval=TRAIN_CONFIG["checkpoint_interval"],
    early_stop_rounds=TRAIN_CONFIG["early_stop_rounds"],
    model_name="Main Training"
)

# --- Post-Training: Monitor Memory Usage ---
print("\nPost-training memory check:")
import psutil
print(f"Memory usage: {psutil.virtual_memory().percent}%")
print(f"Available memory: {psutil.virtual_memory().available / (1024**3):.1f} GB")

In [ ]:
# Declare params again for next task
params = {
    'objective': 'multi:softprob',
    'num_class': 1000,
    'tree_method': 'hist',
    'device': 'cuda',
    'eval_metric': 'mlogloss',
    'eta': best_params.get('learning_rate', 0.1),
    'max_depth': min(best_params.get('max_depth', 6), 4),  # Cap depth at 4
    'subsample': 0.5,  # 50% rows per tree (was 0.8)
    'colsample_bytree': 0.5,  # 50% features per tree (was 0.8)
    'reg_lambda': best_params.get('reg_lambda', 1),
    'random_state': 42,
    'seed': 42,
    'max_bin': 64,
    'grow_policy': 'lossguide',
    'max_leaves': 32  # Limit leaves per tree (caps complexity/VRAM)
}

In [ ]:
# ----------------------------------------------------------
# 6. Evaluation
# ----------------------------------------------------------
print("\n📊 Evaluating model...")

duration = 211.5 # duration from previous run
# Use the clf object returned from the reusable training function
# (No need to load model - it's already available)

# Predictions
val_pred = clf.predict(X_val)
test_pred = clf.predict(X_test)
val_prob = clf.predict_proba(X_val)
test_prob = clf.predict_proba(X_test)

def top_k_accuracy(y_true, y_prob, k=5):
    """Calculate top-k accuracy"""
    top_k = np.argsort(-y_prob, axis=1)[:, :k]
    return np.mean([y_true[i] in top_k[i] for i in range(len(y_true))])

# Calculate metrics
val_acc = accuracy_score(y_val, val_pred)
test_acc = accuracy_score(y_test, test_pred)
val_top5 = top_k_accuracy(y_val, val_prob)
test_top5 = top_k_accuracy(y_test, test_prob)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Validation  - Top-1: {val_acc*100:.2f}% | Top-5: {val_top5*100:.2f}%")
print(f"Test(V2)    - Top-1: {test_acc*100:.2f}% | Top-5: {test_top5*100:.2f}%")
print(f"Training Time: {duration/60:.1f} minutes")
print("="*60)

In [ ]:
# ----------------------------------------------------------
# 7. Feature Importance Visualization
# ----------------------------------------------------------
print("\n📈 Generating feature importance plot...")
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
plt.bar(range(20), importances[indices[:20]], color="#55A868")
plt.title("Top 20 Feature Importances", fontsize=14, fontweight='bold')
plt.xlabel("Feature Index", fontsize=12)
plt.ylabel("Importance", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "feature_importances.png"), dpi=150)
plt.show()

In [ ]:
# ----------------------------------------------------------
# Export Feature Importance Table
# ----------------------------------------------------------
print("\n💾 Exporting feature importance table...")

# Load model if not already in memory
if 'clf' not in locals():
    print("   Loading model from checkpoint...")
    clf = XGBClassifier()
    clf.load_model(CHECKPOINT_PATH)

# Get feature importances
importances = clf.feature_importances_
feature_names = [f'feature_{i}' for i in range(len(importances))]

# Create DataFrame with feature importance data
importance_df = pd.DataFrame({
    'feature_index': range(len(importances)),
    'feature_name': feature_names,
    'importance_score': importances
})

# Sort by importance (descending)
importance_df = importance_df.sort_values('importance_score', ascending=False).reset_index(drop=True)
importance_df['rank'] = importance_df.index + 1

# Reorder columns for better readability
importance_df = importance_df[['rank', 'feature_index', 'feature_name', 'importance_score']]

# Save to CSV
output_path = os.path.join(OUTPUT_DIR, "feature_importances.csv")
importance_df.to_csv(output_path, index=False)

print(f"✅ Feature importance table saved to {output_path}")
print(f"   Total features: {len(importance_df)}")
print(f"\nTop 20 most important features:")
print(importance_df.head(20).to_string(index=False))

In [ ]:
# ----------------------------------------------------------
# 8. Save Results
# ----------------------------------------------------------
results = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "val_top1_accuracy": float(val_acc),
    "val_top5_accuracy": float(val_top5),
    "test_top1_accuracy": float(test_acc),
    "test_top5_accuracy": float(test_top5),
    "training_time_minutes": float(training_meta.get('cumulative_time', 0) / 60),
    "best_iteration": int(training_meta.get('best_iteration', 0)),
    "num_training_samples": int(X_train.shape[0]),
    "num_validation_samples": int(X_custom_val.shape[0]),
    "num_test_samples": int(X_test.shape[0]),
    "hyperparameters": params
}

with open(os.path.join(OUTPUT_DIR, "final_results.json"), 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n💾 Results saved to {OUTPUT_DIR}/final_results.json\n")
for key, value in results.items():
    print(f"{key}: {value}")
print("\n✅ Training pipeline complete!")

In [ ]:
# ============================================================
# 9. PERFORMANCE GAP ANALYSIS
# ============================================================
print("\n" + "="*60)
print("PERFORMANCE GAP ANALYSIS: Val vs Test")
print("="*60)

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from scipy.stats import ks_2samp, wasserstein_distance, chi2_contingency

# Config
SAMPLE_FOR_TSNE = 3000
SAMPLE_FOR_DOMAIN = 50000
TOP_FEAT_SHOW = 10
TOP_CLASS_SHOW = 10

# --- 9.1: Per-Class Accuracy Delta ---
print("\n📊 9.1: Analyzing per-class accuracy differences...")

def per_class_acc(y_true, y_pred):
    classes = np.unique(y_true)
    accs = {}
    for c in classes:
        mask = (y_true == c)
        if np.sum(mask) > 0:
            accs[c] = accuracy_score(y_true[mask], y_pred[mask])
    return accs

acc_val_dict = per_class_acc(y_val, val_pred)
acc_test_dict = per_class_acc(y_test, test_pred)

# Create delta dataframe
classes_all = sorted(list(set(list(acc_val_dict.keys()) + list(acc_test_dict.keys()))))
class_delta_rows = []
for c in classes_all:
    v = acc_val_dict.get(c, np.nan)
    t = acc_test_dict.get(c, np.nan)
    if not np.isnan(v) and not np.isnan(t):
        class_delta_rows.append({
            'class': int(c),
            'val_acc': v,
            'test_acc': t,
            'delta': v - t,
            'abs_delta': abs(v - t)
        })

df_class_delta = pd.DataFrame(class_delta_rows).sort_values('delta', ascending=False).reset_index(drop=True)

print(f"\n🔴 Top {TOP_CLASS_SHOW} classes where Val >> Test (performance drops):")
print(df_class_delta.head(TOP_CLASS_SHOW)[['class', 'val_acc', 'test_acc', 'delta']])

print(f"\n🟢 Top {TOP_CLASS_SHOW} classes where Test >> Val (performance improves):")
print(df_class_delta.tail(TOP_CLASS_SHOW)[['class', 'val_acc', 'test_acc', 'delta']])

df_class_delta.to_csv(os.path.join(OUTPUT_DIR, "class_accuracy_delta.csv"), index=False)

# Summary statistics
avg_val_acc = df_class_delta['val_acc'].mean()
avg_test_acc = df_class_delta['test_acc'].mean()
avg_delta = df_class_delta['delta'].mean()
print(f"\n📈 Summary:")
print(f"   Average Val Accuracy:  {avg_val_acc*100:.2f}%")
print(f"   Average Test Accuracy: {avg_test_acc*100:.2f}%")
print(f"   Average Delta:         {avg_delta*100:.2f}%")

# --- 9.2: Confusion Matrix Analysis ---
print("\n📊 9.2: Confusion matrix analysis for problematic classes...")

top_problem_classes = df_class_delta.nlargest(TOP_CLASS_SHOW, 'abs_delta')['class'].tolist()
print(f"   Analyzing classes: {top_problem_classes}")

def plot_confusion_subset(y_true, y_pred, classes_subset, title, filename):
    mask = np.isin(y_true, classes_subset)
    y_true_sub = y_true[mask]
    y_pred_sub = y_pred[mask]

    unique_classes = sorted(classes_subset)
    label_map = {c: i for i, c in enumerate(unique_classes)}

    y_true_mapped = np.array([label_map[c] for c in y_true_sub])
    y_pred_mapped = np.array([label_map.get(c, -1) for c in y_pred_sub])

    cm = confusion_matrix(y_true_mapped, y_pred_mapped, labels=list(range(len(unique_classes))))

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Predicted Class', fontsize=12)
    plt.ylabel('True Class', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=150)
    plt.show()

plot_confusion_subset(y_val, val_pred, top_problem_classes,
                     "Confusion Matrix (Val) - Top Problem Classes",
                     "cm_val_problem.png")
plot_confusion_subset(y_test, test_pred, top_problem_classes,
                     "Confusion Matrix (Test) - Top Problem Classes",
                     "cm_test_problem.png")

# --- 9.3: Feature Distribution Analysis ---
print("\n📊 9.3: Feature distribution shift analysis...")

# Convert to DataFrame for easier analysis
X_val_df = pd.DataFrame(X_val, columns=[f'f{i}' for i in range(X_val.shape[1])])
X_test_df = pd.DataFrame(X_test, columns=[f'f{i}' for i in range(X_test.shape[1])])

feature_cols = X_val_df.columns.tolist()
ks_stats = []
was_stats = []

print("   Computing KS and Wasserstein statistics...")
for i, col in enumerate(feature_cols):
    if i % 100 == 0:
        print(f"   Progress: {i}/{len(feature_cols)}")

    a = X_val_df[col].values
    b = X_test_df[col].values

    try:
        ks = ks_2samp(a, b).statistic
        wd = wasserstein_distance(a, b)
    except:
        ks = np.nan
        wd = np.nan

    ks_stats.append(ks)
    was_stats.append(wd)

shift_df = pd.DataFrame({
    'feature': feature_cols,
    'ks_statistic': ks_stats,
    'wasserstein_dist': was_stats
}).sort_values('ks_statistic', ascending=False)

print(f"\n🔍 Top {TOP_FEAT_SHOW} features with largest distribution shift:")
print(shift_df.head(TOP_FEAT_SHOW))
shift_df.to_csv(os.path.join(OUTPUT_DIR, "feature_shift_analysis.csv"), index=False)

# Plot top shifted features
top_shift_features = shift_df.head(5)['feature'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, feat in enumerate(top_shift_features[:6]):
    ax = axes[idx]
    ax.hist(X_val_df[feat], bins=50, alpha=0.5, label='Val', density=True, color='blue')
    ax.hist(X_test_df[feat], bins=50, alpha=0.5, label='Test', density=True, color='red')
    ax.set_title(f'{feat}\nKS={shift_df[shift_df.feature==feat].ks_statistic.values[0]:.3f}',
                 fontsize=10)
    ax.legend()
    ax.set_xlabel('Feature Value')
    ax.set_ylabel('Density')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "feature_distribution_shifts.png"), dpi=150)
plt.show()

# --- 9.4: t-SNE Visualization ---
print("\n📊 9.4: t-SNE visualization of feature space...")

n_sample = min(SAMPLE_FOR_TSNE, len(X_val) + len(X_test))
combined_X = np.vstack([X_val, X_test])
combined_labels = np.array(['Val'] * len(X_val) + ['Test'] * len(X_test))

np.random.seed(params['random_state']) # Use random_state from params
sample_idx = np.random.choice(len(combined_X), n_sample, replace=False)
X_sample = combined_X[sample_idx]
labels_sample = combined_labels[sample_idx]

print("   Running PCA...")
pca = PCA(n_components=min(50, X_sample.shape[1]), random_state=params['random_state']) # Use random_state from params
X_pca = pca.fit_transform(X_sample)
print(f"   PCA variance explained (top 10): {pca.explained_variance_ratio_[:10]}")

print("   Running t-SNE (this may take a few minutes)...")
tsne = TSNE(n_components=2, perplexity=30, init='pca', random_state=params['random_state'], n_jobs=-1) # Use random_state from params
X_tsne = tsne.fit_transform(X_pca)

plt.figure(figsize=(10, 8))
for label in ['Val', 'Test']:
    mask = labels_sample == label
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               label=label, alpha=0.6, s=20)
plt.title('t-SNE: Val vs Test Feature Space', fontsize=14, fontweight='bold')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "tsne_val_vs_test.png"), dpi=150)
plt.show()

# --- 9.5: Cluster Analysis ---
print("\n📊 9.5: Cluster distribution analysis...")

k = 5
kmeans = KMeans(n_clusters=k, random_state=params['random_state'], n_init=10) # Use random_state from params
clusters = kmeans.fit_predict(combined_X)

cluster_df = pd.DataFrame({
    'cluster': clusters,
    'domain': combined_labels
})

ct = pd.crosstab(cluster_df['cluster'], cluster_df['domain'])
print("\nCluster distribution:")
print(ct)

chi2, pval, dof, expected = chi2_contingency(ct)
print(f"\n📊 Chi-square test: χ²={chi2:.2f}, p-value={pval:.3g}")
if pval < 0.05:
    print("   ⚠️ Significant difference in cluster distributions (p < 0.05)")
else:
    print("   ✓ No significant difference in cluster distributions")

plt.figure(figsize=(10, 6))
ct.plot(kind='bar', stacked=False, color=['#3498db', '#e74c3c'])
plt.title('Cluster Distribution: Val vs Test', fontsize=14, fontweight='bold')
plt.xlabel('Cluster')
plt.ylabel('Count')
plt.legend(title='Domain')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "cluster_distribution.png"), dpi=150)
plt.show()

# --- 9.6: Generate Analysis Report ---
print("\n📊 9.6: Generating analysis summary report...")

analysis_summary = {
    "performance_gap": {
        "val_accuracy": float(val_acc),
        "test_accuracy": float(test_acc),
        "gap_percentage": float((val_acc - test_acc) * 100),
        "average_class_delta": float(avg_delta)
    },
    "distribution_shift": {
        "mean_ks_statistic": float(shift_df['ks_statistic'].mean()),
        "max_ks_statistic": float(shift_df['ks_statistic'].max()),
        "top_shifted_features": shift_df.head(10)['feature'].tolist(),
        "chi_square_clusters": float(chi2),
        "chi_square_pvalue": float(pval)
    },
    "problematic_classes": {
        "worst_performance_drop": top_problem_classes[:5],
        "avg_delta_worst_classes": float(df_class_delta.head(10)['delta'].mean())
    },
    "hypothesis": []
}

# Generate hypotheses based on analysis
hypotheses = []

if pval < 0.05:
    hypotheses.append("Domain shift detected: Val and Test have different feature distributions")

if shift_df['ks_statistic'].max() > 0.3:
    hypotheses.append(f"Strong feature shift: Feature '{shift_df.iloc[0]['feature']}' shows KS={shift_df.iloc[0]['ks_statistic']:.3f}")

if avg_delta > 0.05:
    hypotheses.append(f"Systematic performance drop: Average delta = {avg_delta*100:.2f}%")

analysis_summary['hypothesis'] = hypotheses

with open(os.path.join(OUTPUT_DIR, "gap_analysis_summary.json"), 'w') as f:
    json.dump(analysis_summary, f, indent=2)

print("\n" + "="*60)
print("ANALYSIS SUMMARY")
print("="*60)
print(f"Performance Gap: {(val_acc - test_acc)*100:.2f}%")
print(f"Distribution Shift: Mean KS = {shift_df['ks_statistic'].mean():.3f}")
print(f"\nHypotheses Generated:")
for i, h in enumerate(hypotheses, 1):
    print(f"  {i}. {h}")
print("="*60)

In [ ]:
# ============================================================
# 10. HYPOTHESIS TESTING & MODEL REFINEMENT
# ============================================================
import os, json, time, gc
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

print("\n" + "="*60)
print("STEP 10: HYPOTHESIS TESTING & MODEL REFINEMENT")
print("="*60)

# Paths
OUTPUT_DIR = os.path.join(BASE_DIR, "results")  # Use existing results directory
DATA_DIR = os.path.join(BASE_DIR, "data", "features")  # Use local features directory
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "xgb_refined_domain_adapted.json")
META_PATH = os.path.join(OUTPUT_DIR, "xgb_refined_meta.json")

# ------------------------------------------------------------
# Load features (num_classes will be determined later from actual labels)
# ------------------------------------------------------------
def load_features_quick(path, nrows=None, dtype='float32'):
    """Load features and labels from CSV.

    Args:
        path: Path to CSV file
        nrows: Number of rows to load (None = load all)
        dtype: Data type for features (use float32 to save memory)

    Returns:
        X: Feature matrix
        y: Label array
    """
    print(f"   Loading {path.split('/')[-1]}...")
    df = pd.read_csv(path, nrows=nrows)
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

    # Extract labels
    y = df['label'].astype('int32').values

    # Extract features (exclude 'label' and 'path' columns)
    feature_cols = [col for col in df.columns if col not in ['label', 'path']]
    X = df[feature_cols].astype(dtype).values

    print(f"      → Loaded {X.shape[0]:,} samples, {X.shape[1]} features, "
          f"{len(np.unique(y))} unique labels (range: {y.min()}-{y.max()})")

    return X, y

print("\n📥 Loading feature datasets...")
X_train, y_train = load_features_quick(
    os.path.join(DATA_DIR, TRAIN_FEATURES_FILE),
    nrows=None  # Load full training set; change to e.g., 200000 if memory constrained
)
X_val, y_val = load_features_quick(
    os.path.join(DATA_DIR, VAL_FEATURES_FILE)
)
X_test, y_test = load_features_quick(
    os.path.join(DATA_DIR, TEST_FEATURES_FILE)
)

print(f"\n✅ Data loaded successfully:")
print(f"   Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"\n📊 Label statistics:")
print(f"   Train: {len(np.unique(y_train)):,} unique classes out of range [{y_train.min()}, {y_train.max()}]")
print(f"   Val:   {len(np.unique(y_val)):,} unique classes out of range [{y_val.min()}, {y_val.max()}]")
print(f"   Test:  {len(np.unique(y_test)):,} unique classes out of range [{y_test.min()}, {y_test.max()}]")

# Free up memory
gc.collect()

print("\n💡 Note: num_classes will be determined from actual labels in the next cell.")

In [ ]:
# ============================================================
# COMPUTE AND SAVE IMPORTANCE WEIGHTS
# ============================================================

print("\n🔬 Computing importance weights (Train ↔ Test)...")

n_samp = min(50000, len(X_train), len(X_test))
idx_train_s = np.random.choice(len(X_train), n_samp, replace=False)
idx_test_s  = np.random.choice(len(X_test),  n_samp, replace=False)

X_dom = np.vstack([X_train[idx_train_s], X_test[idx_test_s]])
y_dom = np.array([0]*n_samp + [1]*n_samp)

dom_clf = LogisticRegression(max_iter=300, solver='saga', n_jobs=-1)
dom_clf.fit(X_dom, y_dom)

p_test = dom_clf.predict_proba(X_train)[:, 1]
eps = 1e-6
weights = p_test / np.clip(1 - p_test, eps, 1)
weights = np.clip(weights, 0.1, 10)

# ✅ Save to your Drive (e.g., under outputs/)
np.save(os.path.join(OUTPUT_DIR, "importance_weights.npy"), weights)

print(f"✅ Importance weights saved — mean={weights.mean():.3f}, range=({weights.min():.2f},{weights.max():.2f})")

In [ ]:
# ============================================================
# LOAD IMPORTANCE WEIGHTS + PROPER LABEL HANDLING
# ============================================================
weights_path = os.path.join(OUTPUT_DIR, "importance_weights.npy")

if os.path.exists(weights_path):
    weights = np.load(weights_path)
    print(f"✅ Loaded importance weights — shape={weights.shape}, mean={weights.mean():.3f}")
else:
    raise FileNotFoundError("❌ importance_weights.npy not found! Run the weight computation cell first.")

print("\nSTART: domain-weighted training\n")

# ============================================================
# ENSURE NUMPY ARRAYS
# ============================================================
X_train, X_val, X_test = map(np.asarray, [X_train, X_val, X_test])
y_train, y_val, y_test = map(lambda y: np.asarray(y).astype(np.int32), [y_train, y_val, y_test])
weights = np.asarray(weights).astype(np.float32)

print("Shapes:", "X_train", X_train.shape, "y_train", y_train.shape, "weights", weights.shape)
print("        ", "X_val", X_val.shape, "y_val", y_val.shape)
print("        ", "X_test", X_test.shape, "y_test", y_test.shape)

# ============================================================
# DETERMINE num_classes (NO REMAPPING - USE ORIGINAL LABELS)
# ============================================================
# Find the maximum label across all datasets to determine num_classes
all_labels = np.concatenate([y_train, y_val, y_test])
num_classes = int(all_labels.max()) + 1  # +1 because labels are 0-indexed

print(f"\n✅ Label Analysis (using ORIGINAL labels, NO remapping):")
print(f"   num_classes = {num_classes} (max label + 1)")
print(f"   Train: {y_train.min()} → {y_train.max()} ({len(np.unique(y_train))} unique classes)")
print(f"   Val:   {y_val.min()} → {y_val.max()} ({len(np.unique(y_val))} unique classes)")
print(f"   Test:  {y_test.min()} → {y_test.max()} ({len(np.unique(y_test))} unique classes)")

# Use original labels directly (no remapping needed)
y_train_mapped = y_train.copy()
y_val_mapped   = y_val.copy()
y_test_mapped  = y_test.copy()

# ============================================================
# SANITY CHECKS
# ============================================================
assert y_train_mapped.min() >= 0 and y_train_mapped.max() < num_classes, \
    f"Train labels out of range! Found [{y_train_mapped.min()}, {y_train_mapped.max()}], expected [0, {num_classes})"
assert y_val_mapped.min() >= 0 and y_val_mapped.max() < num_classes, \
    f"Val labels out of range! Found [{y_val_mapped.min()}, {y_val_mapped.max()}], expected [0, {num_classes})"
assert y_test_mapped.min() >= 0 and y_test_mapped.max() < num_classes, \
    f"Test labels out of range! Found [{y_test_mapped.min()}, {y_test_mapped.max()}], expected [0, {num_classes})"
assert X_train.shape[0] == y_train_mapped.shape[0] == weights.shape[0], \
    f"Dimension mismatch! X_train: {X_train.shape[0]}, y_train: {y_train_mapped.shape[0]}, weights: {weights.shape[0]}"

print(f"✅ All assertions passed!")

# Important note
if len(np.unique(y_train_mapped)) < num_classes:
    print(f"\n⚠️  NOTE: Training subset contains only {len(np.unique(y_train_mapped))} of {num_classes} classes.")
    print(f"   Model will only learn to predict these {len(np.unique(y_train_mapped))} classes.")
    print(f"   Classes not in training will always be predicted incorrectly.")

# ============================================================
# NORMALIZE WEIGHTS
# ============================================================
weights = weights / max(weights.mean(), 1e-6)
weights = np.clip(weights, 0.3, 3.0)
print(f"\n📊 Weights normalized: min={weights.min():.3f}, mean={weights.mean():.3f}, max={weights.max():.3f}")

In [ ]:
# ============================================================
# REFINED TRAINING
# ============================================================

from sklearn.metrics import accuracy_score

# Checkpoint paths for domain-weighted model
CHECKPOINT_PATH_DOMAIN = os.path.join(OUTPUT_DIR, "domain_xgb_checkpoint.json")
META_PATH_DOMAIN = os.path.join(OUTPUT_DIR, "domain_xgb_meta.json")

params_refined = {
    "objective": "multi:softprob",
    "num_class": num_classes,
    "tree_method": "hist",
    "device": DEVICE,  # Use detected device (CUDA/MPS/CPU)
    "eval_metric": "mlogloss",
    "eta": 0.05,
    "max_depth": 6,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "reg_lambda": 1.0,
    "max_bin": 256,  # Use default bin size
    "grow_policy": "lossguide",
    "max_leaves": 64,  # Allow more leaves
    "random_state": 42,
    "seed": 42
}

# --- Train domain-weighted model using reusable function ---
booster_domain, clf_domain, training_meta_domain = train_xgboost_with_checkpointing(
    X_train=X_train, y_train=y_train_mapped,
    X_val=X_val, y_val=y_val_mapped,
    params=params_refined,
    checkpoint_path=CHECKPOINT_PATH_DOMAIN,
    meta_path=META_PATH_DOMAIN,
    sample_weights=weights,  # Domain adaptation weights
    max_rounds=TRAIN_CONFIG["max_rounds"],
    checkpoint_interval=TRAIN_CONFIG["checkpoint_interval"],
    early_stop_rounds=TRAIN_CONFIG["early_stop_rounds"],
    model_name="Domain-Weighted"
)

In [ ]:

# ============================================================
# EVALUATE
# ============================================================
print("\n📊 Evaluating domain-weighted model...")

val_pred = clf_domain.predict(X_val)
test_pred = clf_domain.predict(X_test)

val_acc_dw = accuracy_score(y_val_mapped, val_pred)
test_acc_dw = accuracy_score(y_test_mapped, test_pred)

print(f"📊 Domain-Weighted → Val {val_acc_dw*100:.2f}% | Test {test_acc_dw*100:.2f}% | Gap {(val_acc_dw-test_acc_dw)*100:.2f}%")

# ============================================================
# SAVE MODEL
# ============================================================
model_path = os.path.join(OUTPUT_DIR, "xgb_refined_domain_adapted.json")
clf_domain.save_model(model_path)
print(f"💾 Model saved to {model_path}")

In [ ]:
# Declare variables for next training step
df_class_delta = pd.read_csv(os.path.join(OUTPUT_DIR, "class_accuracy_delta.csv"))
n_classes = 1000

In [ ]:
# ------------------------------------------------------------
# Class-weighted retraining
# ------------------------------------------------------------

from sklearn.metrics import accuracy_score

# Checkpoint paths for class-weighted model
CHECKPOINT_PATH_CLASS = os.path.join(OUTPUT_DIR, "xgb_class_checkpoint.json")
META_PATH_CLASS = os.path.join(OUTPUT_DIR, "xgb_class_meta.json")

# Refined training params (moderate capacity)
params_refined = {
    "objective": "multi:softprob",
    "num_class": n_classes, # Use the determined number of classes
    "tree_method": "hist",
    "device": DEVICE,  # Use detected device (CUDA/MPS/CPU)
    "eval_metric": "mlogloss",
    "eta": 0.05,
    "max_depth": 6,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "reg_lambda": 1.0,
    "random_state": 42,
    "seed": 42,
    "max_bin": 256,  # Use default bin size
    "grow_policy": "lossguide",
    "max_leaves": 64,  # Allow more leaves
}

# Calculate class weights
class_weights = {}
for _, row in df_class_delta.iterrows():
    c = int(row["class"]); delta = row["delta"]
    class_weights[c] = 1.0 + max(0, delta * 3)

sample_weights_class = np.array([class_weights.get(y, 1.0) for y in y_train])
sample_weights_class = sample_weights_class / sample_weights_class.mean()

# --- Train class-weighted model using reusable function ---
booster_class, clf_class, training_meta_class = train_xgboost_with_checkpointing(
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    params=params_refined,
    checkpoint_path=CHECKPOINT_PATH_CLASS,
    meta_path=META_PATH_CLASS,
    sample_weights=sample_weights_class,  # Class rebalancing weights
    max_rounds=TRAIN_CONFIG["max_rounds"],
    checkpoint_interval=TRAIN_CONFIG["checkpoint_interval"],
    early_stop_rounds=TRAIN_CONFIG["early_stop_rounds"],
    model_name="Class-Weighted"
)


In [ ]:
# ============================================================
# EVALUATE
# ============================================================
print("\n📊 Evaluating class-weighted model...")

val_pred_cw = clf_class.predict(X_val)
test_pred_cw = clf_class.predict(X_test)
val_acc_cw = accuracy_score(y_val, val_pred_cw)
test_acc_cw = accuracy_score(y_test, test_pred_cw)
print(f"📊 Class-Weighted → Val {val_acc_cw*100:.2f}% | Test {test_acc_cw*100:.2f}% | Gap {(val_acc_cw-test_acc_cw)*100:.2f}%")

In [ ]:
# ============================================================
# SAVE MODEL
# ============================================================
model_path = os.path.join(OUTPUT_DIR, "xgb_refined_class_weighted.json")
clf_class.save_model(model_path)
print(f"💾 Model saved to {model_path}")

In [ ]:
# If session is stop, use this cell to declare results above
# ------------------------------------------------------------
# RESULT FROM PREVIOUS RUN
# ------------------------------------------------------------

# val_acc = 86.86
# test_acc = 76.47
# val_acc_dw = 88.33
# test_acc_dw = 79.54
# val_acc_cw = 87.75
# test_acc_cw = 78.44

In [ ]:
# ------------------------------------------------------------
# FINAL MODEL COMPARISON
# ------------------------------------------------------------
import pandas as pd
import matplotlib.pyplot as plt
import os

print("\n" + "="*60)
print("FINAL MODEL COMPARISON (Domain vs Class)")
print("="*60)

# Create comparison DataFrame
compare = pd.DataFrame([
    ["Baseline", val_acc, test_acc, val_acc-test_acc],
    ["Domain-Weighted", val_acc_dw, test_acc_dw, val_acc_dw - test_acc_dw],
    ["Class-Weighted", val_acc_cw, test_acc_cw, val_acc_cw - test_acc_cw],
    ["Combined", val_acc_cb, test_acc_cb, val_acc_cb - test_acc_cb],
], columns=["Model", "Val", "Test", "Gap"])

print(compare)

# Save
compare_path = os.path.join(OUTPUT_DIR, "model_refined_comparison.csv")
compare.to_csv(compare_path, index=False)
print(f"\n💾 Comparison saved to {compare_path}")

# Determine best model
best_model_name = compare.iloc[compare["Test"].idxmax()]["Model"]
print(f"\n🏆 Best refined model: {best_model_name}")

# --- Visualization ---
plt.figure(figsize=(6, 4))
bar_width = 0.35
x = range(len(compare))

plt.bar([i - bar_width/2 for i in x], compare["Val"]*100, width=bar_width, label="Validation", alpha=0.8)
plt.bar([i + bar_width/2 for i in x], compare["Test"]*100, width=bar_width, label="Test", alpha=0.8)

plt.xticks(x, compare["Model"])
plt.ylabel("Accuracy (%)")
plt.title("Model Comparison: Domain vs Class Weighting")
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()